In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
chip_data = pd.read_csv("../Data/downloaded/ml_hardware.csv")


In [3]:
chip_data.columns

Index(['Hardware name', 'Manufacturer', 'Type', 'Notes', 'Release date',
       'Release price (USD)', 'Tensor-FP16/BF16 performance (FLOP/s)',
       'FP8 performance (FLOP/s)', 'FP4 performance (FLOP/s)',
       'Memory (bytes)', 'Memory bandwidth (byte/s)',
       'Intranode bandwidth (byte/s)', 'Internode bandwidth (bit/s)',
       'TDP (W)', 'FP64 (double precision) performance (FLOP/s)',
       'FP32 (single precision) performance (FLOP/s)',
       'TF32 (TensorFloat-32) performance (FLOP/s)',
       'FP16 (half precision) performance (FLOP/s)',
       'INT16 performance (OP/s)', 'INT8 performance (OP/s)',
       'INT4 performance (OP/s)', 'Foundry', 'Die Size (mm^2)',
       'Base clock (MHz)', 'Boost clock (MHz)', 'Memory clock (MHz)',
       'Memory bus (bit)', 'Tensor cores', 'Process size (nm)',
       'Transistors (millions)', 'Link to datasheet', 'Source for the price',
       'ML models', 'Last modified', 'Max performance', 'Energy efficiency',
       'Total processing pe

In [4]:

# Attempt to parse the dates with a specified format first, then fall back to general parsing
chip_data['Release Year'] = pd.to_datetime(chip_data['Release date'], format='%m/%d/%y', errors='coerce').dt.year

# For rows where parsing failed, attempt parsing again without a specific format
missing_years = chip_data['Release Year'].isna()
chip_data.loc[missing_years, 'Release Year'] = pd.to_datetime(
	chip_data.loc[missing_years, 'Release date'], errors='coerce'
).dt.year
chip_data['Release Year'] = chip_data['Release Year'].astype('Int64')  # Use Int64 instead of int


In [5]:
# Coerce TDP to integers
chip_data['TDP (W)'] = pd.to_numeric(chip_data['TDP (W)'], errors='coerce').astype('Int64')

# Convert Process Size to numeric
chip_data['Process Size (nm)'] = pd.to_numeric(chip_data['Process size (nm)'], errors='coerce')
chip_data['Die Size (mm^2)'] = pd.to_numeric(chip_data['Die Size (mm^2)'], errors='coerce')
chip_data['TDP (W)/Die Size (mm^2)'] = chip_data['TDP (W)'] / chip_data['Die Size (mm^2)']

# Other metrics
chip_data['Tensor-FP16/BF16 performance (FLOP/s)/TDP(W)'] = chip_data['Tensor-FP16/BF16 performance (FLOP/s)'] / chip_data['TDP (W)']


# Clock throttle gap = (Boost - Base) / Boost   (%) — throttling headroom proxy
chip_data['Clock Throttle Gap (%)'] = (chip_data['Boost clock (MHz)'] - chip_data['Base clock (MHz)']) / chip_data['Boost clock (MHz)'] * 100
chip_data['Boost clock (MHz) - Base Clock (MHz)'] = chip_data['Boost clock (MHz)'] - chip_data['Base clock (MHz)']
# Transistor density = Transistors / Die Size    (MTr/mm²) — scaling pressure
chip_data['Transistor Density (millions/mm²)'] = chip_data['Transistors (millions)'] / chip_data['Die Size (mm^2)']

# Performance per mm² = Tensor Perf / Die Size  — compute density
chip_data['Performance per die area (TFLOP/s/mm²)'] = chip_data['Tensor-FP16/BF16 performance (FLOP/s)'] / chip_data['Die Size (mm^2)']

# Convert Release Date to datetime for plotting
# chip_data['Release Date'] = pd.to_datetime(chip_data['Release Date'], errors='coerce')
# chip_data['Release Year'] = pd.to_datetime(chip_data['Release date'], format='%m/%d/%y', errors='coerce').dt.year
# chip_data['Release Date'] = pd.to_datetime(chip_data['Release date'], format='%m/%d/%y', errors='coerce')

In [6]:
chip_data.head()

,Hardware name,Manufacturer,Type,Notes,Release date,Release price (USD),Tensor-FP16/BF16 performance (FLOP/s),FP8 performance (FLOP/s),FP4 performance (FLOP/s),Memory (bytes),...,Price-performance,ML OP/s,Release Year,Process Size (nm),TDP (W)/Die Size (mm^2),Tensor-FP16/BF16 performance (FLOP/s)/TDP(W),Clock Throttle Gap (%),Boost clock (MHz) - Base Clock (MHz),Transistor Density (millions/mm²),Performance per die area (TFLOP/s/mm²)
0,Meta MTIA 300,Meta,Other,Primarily used for ranking and recommendation ...,2026-03-11,NaN,6.000000e+14,1.200000e+15,NaN,2.160000e+11,...,NaN,1.200000e+15,2026,NaN,<NA>,750000000000.0,NaN,NaN,NaN,NaN
1,Amazon Trainium3,Amazon AWS,GPU,Source for TDP: https://newsletter.semianalysi...,2025-12-02,NaN,6.710000e+14,2.517000e+15,2.517000e+15,NaN,...,NaN,2.517000e+15,2025,NaN,<NA>,958571428571.428589,NaN,NaN,NaN,NaN
2,Google TPU v7 Ironwood,Google,TPU,source for release date: https://www.cnbc.com/...,2025-11-06,NaN,2.307000e+15,4.614000e+15,NaN,1.920000e+11,...,NaN,4.614000e+15,2025,NaN,<NA>,2403125000000.0,NaN,NaN,NaN,NaN
3,Huawei Ascend 920,Huawei,NPU,NaN,2025-10-01,NaN,9.000000e+14,NaN,NaN,NaN,...,NaN,9.000000e+14,2025,6.0,<NA>,<NA>,NaN,NaN,NaN,NaN
4,NVIDIA GB300 (Blackwell Ultra),NVIDIA,GPU,Individual GPUs are sometimes called Blackwell...,2025-08-22,NaN,2.500000e+15,5.000000e+15,1.500000e+16,2.875000e+11,...,NaN,5.000000e+15,2025,4.0,<NA>,1785714285714.285645,NaN,NaN,NaN,NaN


In [7]:
# gpu_data = chip_data[chip_data['Type'] == 'GPU']
# cpu_data = chip_data[chip_data['Type'] == 'CPU']
# gpu_data = gpu_data.dropna(subset=['Release Year'])
# gpu_data['FP32 GFLOPS/TDP (W)'] = gpu_data['FP32 GFLOPS'] / gpu_data['TDP (W)']
# gpu_data = gpu_data.drop(gpu_data[gpu_data['Product']=="Intel Data Center GPU Max Subsystem"].index)

In [8]:
# gpu_data['FP32 (single precision) performance (FLOP/s)/TDP (W)'] = gpu_data['FP32 (single precision) performance (FLOP/s)'] / gpu_data['TDP (W)']



In [9]:
def plot_scatter(gpu_data, x, y, c, hover_info=None):
    """
    Scatter plot with discrete color legend entries.

    Parameters
    ----------
    y : str or [str, str]
        Single column for one y-axis, or [y_left, y_right] for dual y-axes.
        Dual-axis traces use circle (left) and diamond (right) markers.
    """
    import plotly.colors as pc

    dual_axis = isinstance(y, (list, tuple)) and len(y) == 2
    y_left  = y[0] if dual_axis else y
    y_right = y[1] if dual_axis else None

    # ── Discrete color mapping ────────────────────────────────────────────
    c_vals = pd.to_numeric(gpu_data[c], errors='coerce').dropna().astype(float)
    unique_vals = sorted(c_vals.unique())
    n = len(unique_vals)
    val_to_idx = {v: i for i, v in enumerate(unique_vals)}

    sample_pts = [i / (n - 1) for i in range(n)] if n > 1 else [0.5]
    color_list = pc.sample_colorscale('Viridis', sample_pts)

    # Step colorscale: flat color band per category
    discrete_cs = []
    for i, col in enumerate(color_list):
        discrete_cs += [[i / n, col], [(i + 1) / n, col]]

    def val_label(v):
        return str(int(v)) if v == round(v) else str(v)

    # ── Helpers ───────────────────────────────────────────────────────────
    def _filter(cols):
        d = gpu_data.dropna(subset=cols).copy()
        d[c] = pd.to_numeric(d[c], errors='coerce').astype(float)
        return d.dropna(subset=[c])

    def _make_trace(data, y_col, symbol, show_in_legend, legend_group, group_title=None):
        hover = (
            data['Hardware name'].astype(str).values
            if hover_info is None
            else data[hover_info].astype(str).values
        )
        color_vals = data[c].astype(float).map(val_to_idx) / n + 0.5 / n

        kwargs = dict(
            x=data[x], y=data[y_col],
            mode='markers',
            marker=dict(
                size=8,
                color=color_vals,
                colorscale=discrete_cs,
                cmin=0, cmax=1,
                symbol=symbol,
                showscale=False,
            ),
            name=y_col,
            legendgroup=legend_group,
            showlegend=show_in_legend,
            customdata=hover,
            hovertemplate=(
                '<b>%{customdata}</b><br>' +
                x + ': %{x}<br>' +
                y_col + ': %{y}<br>' +
                '<extra></extra>'
            ),
        )
        if group_title:
            kwargs['legendgrouptitle'] = dict(text=group_title)
        return go.Scatter(**kwargs)

    # ── Shared layout helpers ─────────────────────────────────────────────
    ax = dict(showline=True, linewidth=1, linecolor='black',
              mirror=True, showgrid=True, gridcolor='lightgray')
    border = [dict(type='rect', x0=0, x1=1, y0=0, y1=1,
                   xref='paper', yref='paper',
                   line=dict(color='black', width=1))]

    # ── Build figure ──────────────────────────────────────────────────────
    if dual_axis:
        fig = make_subplots(specs=[[{'secondary_y': True}]])
        fig.add_trace(
            _make_trace(_filter([x, y_left, c]),  y_left,  'circle',  True, 'series', group_title='Series'),
            secondary_y=False,
        )
        fig.add_trace(
            _make_trace(_filter([x, y_right, c]), y_right, 'diamond', True, 'series'),
            secondary_y=True,
        )
        fig.update_layout(
            height=500, width=750,
            plot_bgcolor='white', paper_bgcolor='white',
            xaxis=dict(**ax, title=x),
            yaxis=dict(**ax, title=y_left),
            yaxis2=dict(showline=True, linewidth=1, linecolor='black',
                        showgrid=False, title=y_right),
            shapes=border,
        )
    else:
        fig = go.Figure(data=[
            _make_trace(_filter([x, y_left, c]), y_left, 'circle', False, 'data')
        ])
        fig.update_layout(
            height=500, width=600,
            plot_bgcolor='white', paper_bgcolor='white',
            xaxis=dict(**ax, title=x),
            yaxis=dict(**ax, title=y_left),
            shapes=border,
        )

    # ── Discrete color entries in the legend (one square per unique c value)
    for i, val in enumerate(unique_vals):
        kwargs = dict(
            x=[None], y=[None],
            mode='markers',
            marker=dict(size=8, color=color_list[i], symbol='square'),
            name=val_label(val),
            legendgroup=c,
            showlegend=True,
        )
        if i == 0:
            kwargs['legendgrouptitle'] = dict(text=c)
        fig.add_trace(go.Scatter(**kwargs))

    fig.show()

In [10]:
x = 'Release date'
y = 'TDP (W)'
c = 'Process Size (nm)'
plot_scatter(chip_data, x, y, c)

In [11]:
x = 'Release date'
y = 'Tensor-FP16/BF16 performance (FLOP/s)'
c = 'Process Size (nm)'
plot_scatter(chip_data, x, y, c)

In [12]:

x = 'Release date'
y = 'TDP (W)/Die Size (mm^2)'
c = 'Process Size (nm)'
plot_scatter(chip_data, x, y, c)


In [13]:

x = 'Release date'
y = 'Tensor-FP16/BF16 performance (FLOP/s)/TDP(W)'
c = 'Process Size (nm)'
plot_scatter(chip_data, x, y, c)

In [14]:
x = 'Release date'
y = 'Clock Throttle Gap (%)'
c = 'Process Size (nm)'
plot_scatter(chip_data, x, y, c)

In [15]:
x = 'Release date'
y = 'Boost clock (MHz) - Base Clock (MHz)'
c = 'Process Size (nm)'
plot_scatter(chip_data, x, y, c)

In [16]:
x = 'Process Size (nm)'
y = 'Transistor Density (millions/mm²)'
c = 'Release Year'
plot_scatter(chip_data, x, y, c)

In [18]:
x = 'TDP (W)'
y = 'Tensor-FP16/BF16 performance (FLOP/s)'
c = 'Release Year'
plot_scatter(chip_data, x, y, c)

In [20]:
x = 'Memory bandwidth (byte/s)'
y = 'Tensor-FP16/BF16 performance (FLOP/s)'
c = 'Release Year'
plot_scatter(chip_data, x, y, c)

In [21]:
x = 'TDP (W)/Die Size (mm^2)'
y = 'Tensor-FP16/BF16 performance (FLOP/s)/TDP(W)'
c = 'Release Year'
plot_scatter(chip_data, x, y, c)

In [24]:
x = 'TDP (W)'
y = 'Tensor-FP16/BF16 performance (FLOP/s)/TDP(W)'
c = 'Release Year'
plot_scatter(chip_data, x, y, c)

In [23]:
x = 'TDP (W)'
y = 'Tensor-FP16/BF16 performance (FLOP/s)'
c = 'Release Year'
plot_scatter(chip_data, x, y, c)